# LoRA fine-tuning, measured: Qwen2.5-0.5B, baseline vs prompted baseline vs adapter

**Code:** https://github.com/lyftium/lora-finetune-recipe

## Summary
- **Question:** does a LoRA adapter beat simply prompting better, on a narrow text-to-JSON task?
- **Data:** synthetic strings (calendar entries, order lines, sensor readings) with exact JSON labels; 400 train / 150 test from different seeds, de-duplicated against each other. Everything is made up.
- **Method:** Qwen2.5-0.5B-Instruct on a T4. Score the base model with a plain prompt, then with a schema + 3-shot prompt, then train LoRA (r=16 on q/k/v/o, 0.44% of weights, 150 steps, answer-only loss) and score it on the same test set. Metrics: valid JSON and exact match.
- **Result (this run):** plain prompt 0.67 valid / 0.00 exact; schema + 3-shot prompt 1.00 / 1.00; LoRA adapter 1.00 / 1.00 with a ~7.5x shorter prompt (30 vs 223 tokens). On this task I'd ship the prompt; the adapter only wins on cost.
- **What I'd do next:** a harder task where prompting doesn't saturate, more than one seed, and serving the adapter with vLLM to check served outputs match the offline eval.

The recipe in the repo uses a different synthetic task; the training and eval code here follows it closely.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q "peft>=0.13"
import json, random, time, hashlib, gc
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import transformers, peft
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "", "| torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
CFG = dict(base_model="Qwen/Qwen2.5-0.5B-Instruct", r=16, alpha=32, dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
           n_train=400, n_test=150, lr=2e-4, batch_size=8, max_steps=150, max_len=96, max_new_tokens=64, seed=0)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## The synthetic task

In [ ]:
TITLES = ["Standup", "Design review", "1:1", "Planning", "Retro", "Demo"]
def example(rng):
    kind = rng.choice(["event", "order", "sensor"])
    if kind == "event":
        t, d, h, m, dur = rng.choice(TITLES), rng.randint(1, 28), rng.randint(8, 18), rng.choice([0, 15, 30, 45]), rng.choice([15, 30, 45, 60, 90])
        text = f"{t} | 2026-{rng.randint(1,12):02d}-{d:02d} {h:02d}:{m:02d} | {dur}min"
        f = {"title": t, "date": text.split(" | ")[1][:10], "time": f"{h:02d}:{m:02d}", "duration_min": dur}
    elif kind == "order":
        sku, price, qty = f"SKU-{rng.randint(1000, 9999)}", round(rng.uniform(2, 300), 2), rng.randint(1, 12)
        text = f"{sku} x{qty} @ EUR {price:.2f}"
        f = {"sku": sku, "quantity": qty, "unit_price": price, "currency": "EUR"}
    else:
        temp, hum, sid = round(rng.uniform(-10, 40), 1), rng.randint(10, 95), f"S{rng.randint(1, 60):02d}"
        text = f"{sid}: T={temp:.1f}C RH={hum}%"
        f = {"sensor": sid, "temperature_c": temp, "humidity_pct": hum}
    return {"kind": kind, "text": text, "target": json.dumps(f, separators=(",", ":"))}

def make_split(n, seed, exclude=()):
    rng, out, seen = random.Random(seed), [], set(exclude)
    while len(out) < n:
        ex = example(rng)
        if ex["text"] not in seen: seen.add(ex["text"]); out.append(ex)
    return out
test = make_split(CFG["n_test"], CFG["seed"] + 1000)
train = make_split(CFG["n_train"], CFG["seed"], exclude={e["text"] for e in test})
assert not {e["text"] for e in train} & {e["text"] for e in test}
print(len(train), "train /", len(test), "test, no overlap. Test mix:", pd.Series([e["kind"] for e in test]).value_counts().to_dict())
for e in test[:3]: print(f"  {e['text']:<40} -> {e['target']}")

In [ ]:
PROMPT = "Extract the fields as JSON.\nInput: {text}\nJSON:"
SHOTS = [example(random.Random(s)) for s in (11, 12, 13, 14, 15, 16)]
SHOTS = [next(e for e in (example(random.Random(s)) for s in range(100, 999)) if e["kind"] == k and e["text"] not in {t["text"] for t in test}) for k in ("event", "order", "sensor")]
FEWSHOT = ("Extract the fields as JSON. Schemas:\n- calendar entry: title, date, time, duration_min\n- order line: sku, quantity, unit_price, currency\n"
           "- sensor reading: sensor, temperature_c, humidity_pct\n\n" + "".join(f"Input: {s['text']}\nJSON: {s['target']}\n\n" for s in SHOTS) + "Input: {text}\nJSON:")

tok = AutoTokenizer.from_pretrained(CFG["base_model"]); tok.pad_token = tok.pad_token or tok.eos_token; tok.padding_side = "left"

def first_json(t):
    i = t.find("{")
    if i < 0: return None
    try: o, _ = json.JSONDecoder().raw_decode(t[i:])
    except json.JSONDecodeError: return None
    return o if isinstance(o, dict) else None

@torch.no_grad()
def evaluate(model, rows, template, bs=16):
    model.eval(); valid = exact = 0; by = {}; samples = []
    for i in range(0, len(rows), bs):
        ch = rows[i:i + bs]
        enc = tok([template.replace("{text}", r["text"]) for r in ch], return_tensors="pt", padding=True).to(dev)
        out = model.generate(**enc, max_new_tokens=CFG["max_new_tokens"], do_sample=False, pad_token_id=tok.pad_token_id)
        for r, t in zip(ch, tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            p, g = first_json(t), json.loads(r["target"]); ok = p is not None and all(p.get(k) == v for k, v in g.items())
            valid += p is not None; exact += ok; by.setdefault(r["kind"], []).append(ok)
            if len(samples) < 3: samples.append(t.strip().split("\n")[0][:90])
    n = len(rows)
    return {"json_valid": valid / n, "exact_match": exact / n, **{f"exact_{k}": float(np.mean(v)) for k, v in sorted(by.items())}}, samples

## Baselines: the base model, plain prompt and few-shot prompt

In [ ]:
base = AutoModelForCausalLM.from_pretrained(CFG["base_model"]).to(dev)
results = {}
for name, tpl in [("base, plain prompt", PROMPT), ("base, schema + 3-shot prompt", FEWSHOT)]:
    results[name], s = evaluate(base, test, tpl)
    print(name, {k: round(v, 3) for k, v in results[name].items()}); print("   e.g.", s[0])
del base; gc.collect(); torch.cuda.empty_cache() if dev == "cuda" else None

## LoRA training
Only the adapter weights train. The prompt tokens are masked with -100 so the loss is on the answer only.

In [ ]:
torch.manual_seed(CFG["seed"]); random.seed(CFG["seed"])
model = AutoModelForCausalLM.from_pretrained(CFG["base_model"]).to(dev)
model = get_peft_model(model, LoraConfig(r=CFG["r"], lora_alpha=CFG["alpha"], lora_dropout=CFG["dropout"],
                                          target_modules=CFG["target_modules"], bias="none", task_type="CAUSAL_LM"))
total = sum(p.numel() for p in model.parameters()); trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable:,} of {total:,} ({trainable / total:.2%})")

def encode(r):
    p = tok(PROMPT.replace("{text}", r["text"]))["input_ids"]; a = tok(" " + r["target"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"ids": (p + a)[:CFG["max_len"]], "labels": ([-100] * len(p) + a)[:CFG["max_len"]]}
data = [encode(r) for r in train]
def collate(b):
    n = max(len(x["ids"]) for x in b); ids = torch.full((len(b), n), tok.pad_token_id); lab = torch.full((len(b), n), -100); m = torch.zeros((len(b), n), dtype=torch.long)
    for i, x in enumerate(b): k = len(x["ids"]); ids[i, :k] = torch.tensor(x["ids"]); lab[i, :k] = torch.tensor(x["labels"]); m[i, :k] = 1
    return {"input_ids": ids.to(dev), "labels": lab.to(dev), "attention_mask": m.to(dev)}

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG["lr"])
model.train(); losses, step, t0, rng = [], 0, time.perf_counter(), random.Random(CFG["seed"])
while step < CFG["max_steps"]:
    rng.shuffle(data)
    for i in range(0, len(data), CFG["batch_size"]):
        loss = model(**collate(data[i:i + CFG["batch_size"]])).loss; loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); opt.zero_grad()
        losses.append(loss.item()); step += 1
        if step >= CFG["max_steps"]: break
train_sec = time.perf_counter() - t0
print(f"{step} steps in {train_sec:.0f}s, loss {losses[0]:.3f} -> {losses[-1]:.4f}")
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.plot(losses, color="#4C72B0"); ax.set_yscale("log")
ax.set(title="Training loss (answer tokens only)", xlabel="step", ylabel="loss"); plt.tight_layout(); plt.show()

In [ ]:
results["LoRA adapter, plain prompt"], s = evaluate(model, test, PROMPT); print("e.g.", s[0])
n_plain = np.mean([len(tok(PROMPT.replace("{text}", r["text"]))["input_ids"]) for r in test]); n_few = np.mean([len(tok(FEWSHOT.replace("{text}", r["text"]))["input_ids"]) for r in test])
print(f"prompt length: plain {n_plain:.0f} tokens vs few-shot {n_few:.0f} tokens ({n_few / n_plain:.1f}x)")
tab = pd.DataFrame(results).T
display(tab.style.format("{:.2f}"))
fig, ax = plt.subplots(figsize=(8, 3.6)); tab[["json_valid", "exact_match"]].plot.barh(ax=ax, color=["#8DA0CB", "#4C72B0"])
ax.set(xlim=(0, 1.05), title=f"Same {len(test)} held-out examples, greedy decoding"); ax.invert_yaxis(); plt.tight_layout(); plt.show()
json.dump({"config": CFG, "results": results, "trainable_params": trainable, "total_params": total, "train_seconds": round(train_sec, 1),
           "loss_first": losses[0], "loss_last": losses[-1], "test_hash": hashlib.sha256(json.dumps(test).encode()).hexdigest()[:16]}, open("report.json", "w"), indent=2)

## What I take from this

- The plain-prompt baseline gets 0% exact match: it has never seen the schema, so it invents field names and wraps answers in code fences, and about a third of its outputs don't even parse within the token limit.
- The honest result is the second row: with the schema and three examples in the prompt, the same base model already gets every test example right. The adapter matches it (100%), but doesn't beat it.
- So on this task I would ship the prompt, not the adapter. The adapter's only real win here is cost: it gets the same answers from a prompt several times shorter (printed above), which matters at volume or with tight context budgets. On a harder task the gap could go the other way, and this is exactly the check I'd run first.
- Synthetic data, one seed: this shows the method (baselines first, same split, small adapter, traceable report), not a claim about messy real inputs.

Full recipe with tests, configs (incl. a QLoRA 7B config) and run records: https://github.com/lyftium/lora-finetune-recipe